In [163]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [164]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    print(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    for item in result.get('data', []):
        if 'summary' in item:
            normalized_data.append(item)

observed_src = pd.json_normalize(normalized_data)
observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
observed_src.drop_duplicates(subset='indicator', inplace=True)
observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]


Querying owner: HTOC Org


In [165]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,ip,legacyLink,tags.data,hostName,dnsActive,whoisActive,source,address,url,indicator
0,5629499556005874,2025-06-30T12:21:41Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-04T10:38:40Z,3.0,75.0,RITM0594014,...,104.234.115.58,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,104.234.115.58
1,5629499561376167,2025-07-28T17:15:28Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-04T10:37:15Z,3.0,79.0,TASK0902923,...,65.49.1.133,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.133
2,6755399465229497,2025-07-28T17:33:52Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-04T10:35:26Z,3.0,79.0,TASK0902923,...,198.235.24.212,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,198.235.24.212
3,6755399460007805,2025-07-02T12:05:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-04T09:13:12Z,4.0,69.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,190.242.118.68,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1469320, 'name': 'INDICATOR NOTICE 251...",NaN,NaN,NaN,NaN,NaN,NaN,190.242.118.68
4,5629499555099357,2025-06-23T15:22:20Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-04T09:12:44Z,3.0,74.0,TASK0890150,...,5.188.206.194,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,5.188.206.194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1184,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90.0,ACD R&F processed a malspam campaign with a Ne...,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 390199, 'name': 'hhs-2024090901', 'las...",NaN,NaN,NaN,https://www.shorturl.at/,NaN,www.shorturl.at/,www.shorturl.at/
1185,4476331,2023-11-16T13:43:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:44Z,4.0,85.0,NaN,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,http://selligenttier.naylorcampaigns.com,NaN,selligenttier.naylorcampaigns.com,selligenttier.naylorcampaigns.com
1186,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84.0,NaN,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 34822, 'name': 'business email comprom...",NaN,NaN,NaN,NaN,NaN,geo.netsupportsoftware.com/location/loca.asp,geo.netsupportsoftware.com/location/loca.asp
1188,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83.0,NaN,...,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,https://google,NaN,google,google


In [166]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250804.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250803.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250802.csv']
Loaded data from 3 files.


In [167]:
import warnings

# Extract API-tagged indicators
all_filtered = []
cutoff = pd.Timestamp.utcnow()

for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    if isinstance(tags_data, list):
        tags_df = pd.json_normalize(tags_data)
        tags_df['name'] = tags_df['name'].str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)
        api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)]
        if not api_tags.empty:
            all_tags_list = tags_df['name'].astype(str).tolist()
            api_tags['name'] = api_tags['name'].astype(str)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved', 'webLink','rating', 'confidence', 'id']:
                api_tags[col] = row.get(col)
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)
            all_filtered.append(api_tags)
            warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
filtered_tags = pd.concat(all_filtered, ignore_index=True)
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Prepare for matching
filtered_tags['OpDiv'] = filtered_tags['name'].str.replace(' Splunk API', '', regex=False)
obs_subset = observed_data_df[['indicator', 'OpDiv', 'obs_date']].drop_duplicates()
recent_tags = pd.merge(filtered_tags, obs_subset, left_on=['summary', 'OpDiv'], right_on=['indicator', 'OpDiv'], how='inner')

# Filter to recent
cutoff_naive = cutoff.tz_convert(None)
recent_tags = recent_tags[
    (recent_tags['lastObserved'] >= cutoff - timedelta(hours=24)) &
    (pd.to_datetime(recent_tags['obs_date'], errors='coerce') >= cutoff_naive - timedelta(days=1))
].copy()

# Partner processing
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))])
    .reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')
recent_tags.drop_duplicates(subset='summary', inplace=True)
recent_tags


,id,name,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,...,tactics.data,tactics.count,platforms.data,platforms.count,OpDiv,indicator,obs_date,partner,partner_count,partners
0,5629499556005874,DHA Splunk API,2025-08-04T03:37:07Z,RITM0594014,104.234.115.58,182726,Address,2025-06-30 12:21:41+00:00,2025-08-04T10:38:40Z,2025-08-04 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,104.234.115.58,2025-08-04,DHA,3,"DHA, HHS, VA"
3,5629499561376167,OS Splunk API,2025-08-04T03:36:52Z,TASK0902923,65.49.1.133,41809,Address,2025-07-28 17:15:28+00:00,2025-08-04T10:37:15Z,2025-08-04 00:00:00+00:00,...,NaN,NaN,NaN,NaN,OS,65.49.1.133,2025-08-04,OS,7,"CDC, CMS, FDA, IHS, NIH, OS, VA"
10,6755399465229497,DHA Splunk API,2025-08-04T03:37:07Z,TASK0902923,198.235.24.212,948380,Address,2025-07-28 17:33:52+00:00,2025-08-04T10:35:26Z,2025-08-04 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,198.235.24.212,2025-08-04,DHA,8,"CDC, CMS, DHA, FDA, IHS, NIH, OS, VA"
18,6755399460007805,CMS Splunk API,2025-08-04T09:13:12Z,"Details\nIn late June 2025, Iran-aligned hackt...",190.242.118.68,1197,Address,2025-07-02 12:05:32+00:00,2025-08-04T09:13:12Z,2025-08-04 00:00:00+00:00,...,NaN,NaN,NaN,NaN,CMS,190.242.118.68,2025-08-04,CMS,1,CMS
19,5629499555099357,OS Splunk API,2025-08-04T03:36:52Z,TASK0890150,5.188.206.194,101321,Address,2025-06-23 15:22:20+00:00,2025-08-04T09:12:44Z,2025-08-04 00:00:00+00:00,...,NaN,NaN,NaN,NaN,OS,5.188.206.194,2025-08-04,OS,4,"CMS, NIH, OS, VA"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3884,5263212,NIH Splunk API,2025-08-04T09:12:44Z,CSO received an email from CSIRC regarding a p...,hcmiu.edu.vn,269,Host,2025-01-22 18:05:29+00:00,2025-05-30T16:33:10Z,2025-08-04 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NIH,hcmiu.edu.vn,2025-08-04,NIH,1,NIH
3885,5263312,NIH Splunk API,2025-08-04T09:12:44Z,socgholish Fake Update C2 malware\nA user was ...,r3.o.lencr.org,8942,Host,2025-01-22 18:43:24+00:00,2025-05-30T15:34:22Z,2025-08-04 00:00:00+00:00,...,NaN,NaN,NaN,NaN,NIH,r3.o.lencr.org,2025-08-04,NIH,1,NIH
3886,5629499542023970,DHA Splunk API,2025-08-04T03:37:07Z,NaN,chaturbate.com,7053,Stripped URL,2025-05-24 02:57:09+00:00,2025-05-27T08:35:54Z,2025-08-04 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,chaturbate.com,2025-08-04,DHA,1,DHA
3887,5629499542023958,DHA Splunk API,2025-08-04T03:37:07Z,NaN,fmovies.co,182,Stripped URL,2025-05-24 02:50:36+00:00,2025-05-27T05:28:51Z,2025-08-04 00:00:00+00:00,...,NaN,NaN,NaN,NaN,DHA,fmovies.co,2025-08-04,DHA,1,DHA


In [168]:
# Enrich only final filtered indicators
indicator_values = recent_tags['summary'].dropna().unique().tolist()
enriched_results = []

print(f"Enriching {len(indicator_values)} indicators with DomainTools and VirusTotalV3...")

for value in indicator_values:
    try:
        # Use the indicator *value*, not the ID
        encoded_value = urllib.parse.quote(value)
        enrich_url = f'/v3/indicators/{encoded_value}/enrich?type=Shodan&type=VirusTotalV3'
        ro.set_http_method('POST')
        ro.set_request_uri(enrich_url)
        ro.set_body({})
        enrich_response = tc.api_request(ro)

        if enrich_response.status_code == 200:
            enrich_data = enrich_response.json()
            enrich_data['summary'] = value  # Save the value as key
            enriched_results.append(enrich_data)
    except Exception as e:
        continue

if enriched_results:
    df_enriched = pd.json_normalize(enriched_results)
    df_enriched = pd.json_normalize(enriched_results)
    recent_tags = recent_tags.merge(df_enriched, on='summary', how='left')
    # Unnest 'data.enrichment.data' (list of dicts) into separate columns
if 'data.enrichment.data' in recent_tags.columns:
    enrichment_df = pd.json_normalize(
        recent_tags['data.enrichment.data'].dropna().explode()
    )
    enrichment_df.index = recent_tags['data.enrichment.data'].dropna().explode().index
    enrichment_cols = [col for col in enrichment_df.columns if col not in recent_tags.columns]
    # Join enrichment columns back to recent_tags
    recent_tags = recent_tags.join(enrichment_df[enrichment_cols], how='left')

    # Keep only records with vtMaliciousCount > 10
    recent_tags = recent_tags[recent_tags['vtMaliciousCount'] > 10]

    print(f"Successfully enriched and merged {len(df_enriched)} indicators.")
else:
    print("No enrichment data retrieved.")


Enriching 817 indicators with DomainTools and VirusTotalV3...


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host synaptris.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host cdcfoundation.org cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host link.mail.beehiiv.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host pdf.ac cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host yourpensionmeeting.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response

Successfully enriched and merged 795 indicators.


In [169]:
import numpy as np

# Unnest the 'data.enrichment.data' column into separate columns for each enrichment type

def extract_enrichment(row):
    """Extracts enrichment fields from the list of dicts in 'data.enrichment.data'."""
    enrichments = row.get('data.enrichment.data')
    result = {}
    if isinstance(enrichments, list):
        for enrich in enrichments:
            enrich_type = enrich.get('type')
            if enrich_type == 'Shodan':
                # Flatten Shodan fields
                for key in ['hostNames', 'domains', 'tags', 'country', 'city', 'isp', 'asn', 'org', 'openPorts']:
                    result[f'shodan_{key}'] = enrich.get(key, np.nan)
            elif enrich_type == 'VirusTotal':
                result['vtMaliciousCount'] = enrich.get('vtMaliciousCount', np.nan)
    return pd.Series(result)

# Apply extraction to recent_tags
enrichment_expanded = recent_tags.apply(extract_enrichment, axis=1)
recent_tags = pd.concat([recent_tags, enrichment_expanded], axis=1)

recent_tags = recent_tags.rename(columns={
    'indicator': 'Indicator',
    'vtMaliciousCount': 'Malicious Score/Count',
    'obs_date': 'Observation Date',
    'shodan_asn': 'ASN',
    'rating': 'ThreatAssessRating',
    'confidence': 'ThreatAssessConfidence',
    'shodan_city': 'City',
    'shodan_country': 'Country',
    'data.legacyLink': 'ThreatConnect Link',
    'partners': 'Partners'
})

# Now select only the columns you want, after renaming
recent_tags = recent_tags[
    [
        'Indicator',
        'Malicious Score/Count',
        'Observation Date',
        'ASN',
        'ThreatAssessRating',
        'ThreatAssessConfidence',
        'City',
        'Country',
        'ThreatConnect Link',
        'Partners'
    ]
]

# Remove duplicate columns by keeping only the first occurrence
recent_tags = recent_tags.loc[:, ~recent_tags.columns.duplicated()]
recent_tags = recent_tags.fillna('unknown')
recent_tags = recent_tags.drop_duplicates()
recent_tags

,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
0,104.234.115.58,11.0,2025-08-04,unknown,3.0,75.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"DHA, HHS, VA"
5,125.124.50.87,13.0,2025-08-04,AS58461,3.0,77.0,Ningbo,China,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, IHS, NIH, OS"
13,205.210.31.41,11.0,2025-08-04,unknown,3.0,79.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
17,198.235.24.253,11.0,2025-08-04,AS396982,3.0,79.0,Antwerpen,Belgium,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
18,198.235.24.79,12.0,2025-08-04,AS396982,3.0,79.0,Changhua,Taiwan,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
...,...,...,...,...,...,...,...,...,...,...
773,199.45.154.180,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, NIH, OS, VA"
775,199.45.154.186,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"
801,199.45.154.178,13.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, OS, VA"
802,199.45.154.187,12.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, NIH, OS, VA"


In [170]:
from IPython.display import display

# Get all unique partners (split by comma and flatten)
all_partners = set(
    p.strip()
    for partners in recent_tags['Partners'].dropna().unique()
    for p in partners.split(',')
)

# Build buckets: each partner gets all rows where it appears in the partners column
partner_buckets = {
    partner: recent_tags[recent_tags['Partners'].str.contains(rf'\b{partner}\b', na=False)]
    for partner in all_partners
}

# Show each partner's dataframe as a table
for partner, df in partner_buckets.items():
    print(f"Partner: {partner} ({len(df)} records)")
    display(df)

Partner: FDA (83 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
5,125.124.50.87,13.0,2025-08-04,AS58461,3.0,77.0,Ningbo,China,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, IHS, NIH, OS"
13,205.210.31.41,11.0,2025-08-04,unknown,3.0,79.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
17,198.235.24.253,11.0,2025-08-04,AS396982,3.0,79.0,Antwerpen,Belgium,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
18,198.235.24.79,12.0,2025-08-04,AS396982,3.0,79.0,Changhua,Taiwan,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
19,198.235.24.176,12.0,2025-08-04,AS396982,3.0,79.0,Antwerpen,Belgium,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, NIH, OS, VA"
...,...,...,...,...,...,...,...,...,...,...
752,65.49.1.45,12.0,2025-08-04,AS6939,3.0,80.0,San Francisco,United States,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"
773,199.45.154.180,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, NIH, OS, VA"
775,199.45.154.186,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"
801,199.45.154.178,13.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, OS, VA"


Partner: VA (98 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
0,104.234.115.58,11.0,2025-08-04,unknown,3.0,75.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"DHA, HHS, VA"
13,205.210.31.41,11.0,2025-08-04,unknown,3.0,79.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
17,198.235.24.253,11.0,2025-08-04,AS396982,3.0,79.0,Antwerpen,Belgium,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
18,198.235.24.79,12.0,2025-08-04,AS396982,3.0,79.0,Changhua,Taiwan,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
19,198.235.24.176,12.0,2025-08-04,AS396982,3.0,79.0,Antwerpen,Belgium,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, NIH, OS, VA"
...,...,...,...,...,...,...,...,...,...,...
752,65.49.1.45,12.0,2025-08-04,AS6939,3.0,80.0,San Francisco,United States,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"
773,199.45.154.180,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, NIH, OS, VA"
775,199.45.154.186,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"
801,199.45.154.178,13.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, OS, VA"


Partner: NIH (19 records)


...

,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
0,104.234.115.58,11.0,2025-08-04,unknown,3.0,75.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"DHA, HHS, VA"
13,205.210.31.41,11.0,2025-08-04,unknown,3.0,79.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
17,198.235.24.253,11.0,2025-08-04,AS396982,3.0,79.0,Antwerpen,Belgium,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
18,198.235.24.79,12.0,2025-08-04,AS396982,3.0,79.0,Changhua,Taiwan,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
19,198.235.24.176,12.0,2025-08-04,AS396982,3.0,79.0,Antwerpen,Belgium,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, NIH, OS, VA"
...,...,...,...,...,...,...,...,...,...,...
752,65.49.1.45,12.0,2025-08-04,AS6939,3.0,80.0,San Francisco,United States,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"
773,199.45.154.180,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, NIH, OS, VA"
775,199.45.154.186,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"
801,199.45.154.178,13.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, OS, VA"


Partner: CMS (86 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
5,125.124.50.87,13.0,2025-08-04,AS58461,3.0,77.0,Ningbo,China,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, IHS, NIH, OS"
13,205.210.31.41,11.0,2025-08-04,unknown,3.0,79.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
17,198.235.24.253,11.0,2025-08-04,AS396982,3.0,79.0,Antwerpen,Belgium,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
18,198.235.24.79,12.0,2025-08-04,AS396982,3.0,79.0,Changhua,Taiwan,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
19,198.235.24.176,12.0,2025-08-04,AS396982,3.0,79.0,Antwerpen,Belgium,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, NIH, OS, VA"
...,...,...,...,...,...,...,...,...,...,...
773,199.45.154.180,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, NIH, OS, VA"
775,199.45.154.186,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"
801,199.45.154.178,13.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, OS, VA"
802,199.45.154.187,12.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, NIH, OS, VA"


Partner: OS (82 records)


,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
5,125.124.50.87,13.0,2025-08-04,AS58461,3.0,77.0,Ningbo,China,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, IHS, NIH, OS"
13,205.210.31.41,11.0,2025-08-04,unknown,3.0,79.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
17,198.235.24.253,11.0,2025-08-04,AS396982,3.0,79.0,Antwerpen,Belgium,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
18,198.235.24.79,12.0,2025-08-04,AS396982,3.0,79.0,Changhua,Taiwan,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, OS, VA"
19,198.235.24.176,12.0,2025-08-04,AS396982,3.0,79.0,Antwerpen,Belgium,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, IHS, NIH, OS, VA"
...,...,...,...,...,...,...,...,...,...,...
752,65.49.1.45,12.0,2025-08-04,AS6939,3.0,80.0,San Francisco,United States,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"
773,199.45.154.180,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, NIH, OS, VA"
775,199.45.154.186,11.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CMS, DHA, FDA, HHS, IHS, OS, VA"
801,199.45.154.178,13.0,2025-08-04,unknown,3.0,80.0,unknown,unknown,https://hvs.threatconnect.com/auth/indicators/...,"CDC, CMS, DHA, FDA, HHS, IHS, OS, VA"


In [171]:
import os
import re
import time
from datetime import datetime
import pandas as pd
import win32com.client as win32

Tippers_Path = r'Z:\HTOC\HTOC Reports\Tippers'
os.makedirs(Tippers_Path, exist_ok=True)

# Add today's date to the filename
today_str = datetime.utcnow().strftime("%Y%m%d")
excel_path = os.path.join(Tippers_Path, f"tippers_by_partner_{today_str}.xlsx")

with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
    for partner, df in partner_buckets.items():
        safe_partner = re.sub(r'[^a-zA-Z0-9_-]', '_', partner)[:31]

        for col in df.select_dtypes(include=['datetimetz']).columns:
            df[col] = df[col].dt.tz_localize(None)

        df.to_excel(writer, sheet_name=safe_partner, index=False)

        worksheet = writer.sheets[safe_partner]
        worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)
        worksheet.freeze_panes(1, 0)

        for i, col in enumerate(df.columns):
            if not df.empty:
                col_lens = df[col].fillna("").astype(str).apply(len)
                max_len = max(col_lens.max(), len(str(col)))
            else:
                max_len = len(str(col))
            worksheet.set_column(i, i, min(max_len + 2, 50))

print(f"Excel file with partner tabs saved to: {excel_path}")



Excel file with partner tabs saved to: Z:\HTOC\HTOC Reports\Tippers\tippers_by_partner_20250804.xlsx
